# LLM Model Evaluation & Finance Ops Briefing Assistant

**Use case:** Build a small, end-to-end LLM workflow that compares OpenAI-style and Anthropic-style model outputs for AI infrastructure / finance decision support.

This notebook is an original project. It uses synthetic data only and does not reproduce course lab text, datasets, outputs, images, or prompts.

## What this project demonstrates

1. Prompt design for business-quality structured output
2. Optional API wrappers for OpenAI and Anthropic
3. A mock LLM mode so the notebook runs without API keys
4. Synthetic AI infrastructure / finance request data
5. Lightweight retrieval over internal policy snippets
6. Structured JSON extraction and validation
7. Basic model evaluation rubric
8. Cost and latency comparison template
9. Executive summary generation

## Why this is relevant to OpenAI / Anthropic-style work

This project is not about building a toy chatbot. It simulates a realistic workflow that AI product, finance, infra, and strategy teams could care about:

- Which model is good enough for a business workflow?
- How do we compare quality, cost, latency, and safety?
- How do we turn messy requests into structured decisions?
- How do we use retrieval to ground model outputs in policy?
- How do we create repeatable evals instead of relying on vibes?


## 1. Setup

The notebook runs in three modes:

| Mode | What it does |
|---|---|
| `mock` | Runs without API keys using deterministic sample outputs |
| `openai` | Calls OpenAI if `OPENAI_API_KEY` and `OPENAI_MODEL` are set |
| `anthropic` | Calls Anthropic if `ANTHROPIC_API_KEY` and `ANTHROPIC_MODEL` are set |

For a public GitHub repo, do **not** hard-code API keys. Use environment variables.


In [ ]:
# Optional installs.
# In GitHub this cell is just documentation; in Colab/Jupyter you can uncomment if needed.
# %pip install -q pandas openai anthropic

import os
import json
import time
import math
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd


## 2. Synthetic Dataset

The dataset below is fully synthetic.  
It represents internal requests that a finance / infra strategy team might evaluate.


In [ ]:
requests = [
    {
        "request_id": "REQ-001",
        "team": "Research Platform",
        "request_type": "training_cluster_expansion",
        "business_context": (
            "The research team wants to reserve additional accelerator capacity for a 10-week "
            "frontier training experiment. The team expects faster iteration cycles and reduced queue time."
        ),
        "technical_context": (
            "Requested capacity: 256 accelerators for 10 weeks. Current utilization averages 68%, "
            "but peak utilization regularly exceeds 92%. Data pipeline has had intermittent bottlenecks."
        ),
        "financial_context": (
            "Estimated incremental run-rate cost is $1.8M for the experiment window. "
            "No approved annual budget uplift yet. Opportunity cost is delaying two smaller eval projects."
        ),
        "risk_context": (
            "Risks include underutilization, training job failure, unclear success metric, and capacity lock-in."
        ),
    },
    {
        "request_id": "REQ-002",
        "team": "Applied AI Product",
        "request_type": "inference_cost_reduction",
        "business_context": (
            "A customer-facing assistant has grown quickly. Leadership wants to reduce inference cost "
            "without hurting response quality or user trust."
        ),
        "technical_context": (
            "Traffic is 18M requests/month. Current routing sends almost all traffic to the largest model. "
            "Roughly 55% of requests are simple classification or short summarization."
        ),
        "financial_context": (
            "Monthly inference cost is $740K and growing 11% month-over-month. Gross margin pressure is increasing."
        ),
        "risk_context": (
            "Risks include degraded answer quality, increased latency, user dissatisfaction, and incorrect routing."
        ),
    },
    {
        "request_id": "REQ-003",
        "team": "Enterprise Solutions",
        "request_type": "regulated_customer_deployment",
        "business_context": (
            "A regulated enterprise customer wants a private deployment for document analysis. "
            "The deal is strategically important but has strict data handling requirements."
        ),
        "technical_context": (
            "Documents may contain confidential financial and employee data. The customer requires audit logs, "
            "role-based access control, and no training on customer data."
        ),
        "financial_context": (
            "Potential ARR is $4.5M. Expected implementation cost is $900K in the first year. "
            "Support burden may be higher than standard deployments."
        ),
        "risk_context": (
            "Risks include privacy breach, unclear data retention policy, compliance failure, and custom support scope creep."
        ),
    },
    {
        "request_id": "REQ-004",
        "team": "Safety Evaluations",
        "request_type": "eval_pipeline_investment",
        "business_context": (
            "The safety team wants to build a more automated evaluation pipeline before launching a more capable model."
        ),
        "technical_context": (
            "Current evals are partly manual. Proposed system includes automated scenario generation, "
            "LLM-as-judge scoring, human review queues, and regression tracking."
        ),
        "financial_context": (
            "Estimated build cost is $650K plus two dedicated engineers for one quarter. "
            "The investment may reduce launch risk and repeated manual review cycles."
        ),
        "risk_context": (
            "Risks include judge model bias, false confidence, insufficient coverage, and evaluation gaming."
        ),
    },
]

df_requests = pd.DataFrame(requests)
df_requests


## 3. Synthetic Policy Knowledge Base

This is a mini retrieval corpus.  
In a real company project, these could be policy docs, launch review docs, finance guardrails, procurement rules, or internal AI safety standards.

For a public portfolio project, keep it synthetic.


In [ ]:
policy_docs = [
    {
        "doc_id": "POL-001",
        "title": "AI Capacity Investment Guardrails",
        "text": (
            "Large training capacity requests should define success metrics before approval. "
            "Requests should include expected utilization, fallback plan, checkpointing plan, and opportunity cost. "
            "Capacity reservations above 8 weeks require finance review and leadership sign-off."
        ),
    },
    {
        "doc_id": "POL-002",
        "title": "Inference Cost Optimization Policy",
        "text": (
            "High-volume inference workloads should use model routing when simple requests can be handled "
            "by smaller or cheaper models. Quality must be monitored with regression evals, latency metrics, "
            "and user-impact guardrails."
        ),
    },
    {
        "doc_id": "POL-003",
        "title": "Customer Data Handling Standard",
        "text": (
            "Customer data must not be used for model training unless explicitly approved by contract. "
            "Private deployments require audit logging, access controls, retention limits, and incident response plans."
        ),
    },
    {
        "doc_id": "POL-004",
        "title": "Evaluation Readiness Standard",
        "text": (
            "Model launches should include automated and human evaluation coverage. LLM-as-judge may be used "
            "as a scalable signal but should be calibrated against human review and monitored for bias."
        ),
    },
]

df_policy = pd.DataFrame(policy_docs)
df_policy


## 4. Lightweight Retrieval

This simple retrieval function uses keyword overlap.  
It is intentionally dependency-light so the notebook works anywhere.

A production version would use embeddings, vector search, reranking, access control, and freshness checks.


In [ ]:
def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9_]+", text.lower())


def retrieve_policies(query: str, policy_docs: List[Dict[str, str]], top_k: int = 2) -> List[Dict[str, Any]]:
    query_terms = set(tokenize(query))
    scored = []

    for doc in policy_docs:
        doc_terms = set(tokenize(doc["title"] + " " + doc["text"]))
        overlap = len(query_terms.intersection(doc_terms))
        score = overlap / max(1, len(query_terms))
        scored.append({**doc, "score": round(score, 3)})

    return sorted(scored, key=lambda x: x["score"], reverse=True)[:top_k]


sample_query = df_requests.loc[0, "business_context"] + " " + df_requests.loc[0, "risk_context"]
retrieve_policies(sample_query, policy_docs)


## 5. Prompt Template

The prompt asks the model to act like a finance and AI infrastructure analyst.

The output is intentionally constrained to JSON so that downstream validation and scoring are possible.


In [ ]:
SYSTEM_INSTRUCTION = """
You are an AI infrastructure finance and strategy analyst.
Your job is to convert messy technical/business requests into structured investment review notes.

Be concise, specific, and decision-oriented.
Do not invent facts. If information is missing, state what is missing.
Return valid JSON only.
""".strip()


def build_analysis_prompt(request: Dict[str, str], retrieved_docs: List[Dict[str, Any]]) -> str:
    policy_context = "\n\n".join(
        [
            f"[{doc['doc_id']}] {doc['title']}: {doc['text']}"
            for doc in retrieved_docs
        ]
    )

    prompt = f"""
Analyze the following AI infrastructure / finance request.

REQUEST:
request_id: {request['request_id']}
team: {request['team']}
request_type: {request['request_type']}

business_context:
{request['business_context']}

technical_context:
{request['technical_context']}

financial_context:
{request['financial_context']}

risk_context:
{request['risk_context']}

RELEVANT POLICY CONTEXT:
{policy_context}

Return JSON with this exact schema:
{{
  "request_id": "...",
  "one_sentence_summary": "...",
  "decision_recommendation": "approve | approve_with_conditions | defer | reject",
  "primary_value_driver": "...",
  "key_risks": ["...", "...", "..."],
  "missing_information": ["...", "..."],
  "finance_questions": ["...", "...", "..."],
  "technical_questions": ["...", "...", "..."],
  "safety_or_compliance_questions": ["...", "..."],
  "recommended_next_step": "...",
  "confidence": "low | medium | high"
}}
""".strip()
    return prompt


example_request = df_requests.iloc[0].to_dict()
retrieved = retrieve_policies(
    " ".join(str(v) for v in example_request.values()),
    policy_docs,
    top_k=2,
)
print(build_analysis_prompt(example_request, retrieved)[:2000])


## 6. LLM Provider Wrappers

This section gives you a clean abstraction layer.

- `mock` mode works without API keys and is enough for GitHub.
- `openai` mode uses the OpenAI Python SDK if configured.
- `anthropic` mode uses the Anthropic Python SDK if configured.

Do not store API keys in notebooks or GitHub.


In [ ]:
@dataclass
class LLMResult:
    provider: str
    model: str
    text: str
    latency_seconds: float
    input_chars: int
    output_chars: int


def mock_llm_response(prompt: str, request_id: str) -> str:
    """Deterministic mock output so the notebook runs without paid APIs."""
    recommendations = {
        "REQ-001": "approve_with_conditions",
        "REQ-002": "approve_with_conditions",
        "REQ-003": "defer",
        "REQ-004": "approve_with_conditions",
    }
    rec = recommendations.get(request_id, "defer")

    return json.dumps(
        {
            "request_id": request_id,
            "one_sentence_summary": "This request may create strategic value but requires clearer operating guardrails before full approval.",
            "decision_recommendation": rec,
            "primary_value_driver": "Improved AI capability, operating efficiency, or enterprise readiness.",
            "key_risks": [
                "Unclear success metric",
                "Cost or capacity commitment may exceed validated demand",
                "Operational or compliance controls may be incomplete",
            ],
            "missing_information": [
                "Defined success metric",
                "Owner for post-launch monitoring",
                "Fallback plan if quality, cost, or utilization misses target",
            ],
            "finance_questions": [
                "What is the expected cost range under base, upside, and downside scenarios?",
                "What budget owner absorbs the incremental cost?",
                "What business metric justifies the investment?",
            ],
            "technical_questions": [
                "What is the expected utilization or request mix?",
                "What are the latency, reliability, and quality guardrails?",
                "What is the rollback or fallback plan?",
            ],
            "safety_or_compliance_questions": [
                "Does the request involve sensitive or customer data?",
                "What audit, access control, and monitoring requirements apply?",
            ],
            "recommended_next_step": "Approve a scoped pilot or defer full approval until success metrics, risk owners, and monitoring plans are defined.",
            "confidence": "medium",
        },
        indent=2,
    )


def call_openai(prompt: str, system_instruction: str, model: str) -> str:
    """
    OpenAI API wrapper.

    Requires:
    - pip install openai
    - export OPENAI_API_KEY=...
    - export OPENAI_MODEL=...
    """
    from openai import OpenAI

    client = OpenAI()
    response = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": prompt},
        ],
    )
    return response.output_text


def call_anthropic(prompt: str, system_instruction: str, model: str) -> str:
    """
    Anthropic API wrapper.

    Requires:
    - pip install anthropic
    - export ANTHROPIC_API_KEY=...
    - export ANTHROPIC_MODEL=...
    """
    import anthropic

    client = anthropic.Anthropic()
    message = client.messages.create(
        model=model,
        max_tokens=1200,
        system=system_instruction,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    # Anthropic message content is a list of content blocks.
    return "".join(
        block.text for block in message.content
        if getattr(block, "type", None) == "text"
    )


def run_llm(provider: str, prompt: str, request_id: str, system_instruction: str = SYSTEM_INSTRUCTION) -> LLMResult:
    provider = provider.lower().strip()
    start = time.time()

    if provider == "mock":
        model = "mock-analyst-v1"
        text = mock_llm_response(prompt, request_id)

    elif provider == "openai":
        model = os.getenv("OPENAI_MODEL")
        if not model:
            raise ValueError("Set OPENAI_MODEL before using provider='openai'.")
        text = call_openai(prompt, system_instruction, model)

    elif provider == "anthropic":
        model = os.getenv("ANTHROPIC_MODEL")
        if not model:
            raise ValueError("Set ANTHROPIC_MODEL before using provider='anthropic'.")
        text = call_anthropic(prompt, system_instruction, model)

    else:
        raise ValueError("provider must be one of: mock, openai, anthropic")

    latency = time.time() - start

    return LLMResult(
        provider=provider,
        model=model,
        text=text,
        latency_seconds=round(latency, 3),
        input_chars=len(prompt),
        output_chars=len(text),
    )


## 7. Run Analysis for One Request

Start with `mock` mode so the notebook runs on GitHub / local Jupyter without API keys.

To use real APIs, change `PROVIDERS` to include `openai` and/or `anthropic` after setting environment variables.


In [ ]:
PROVIDERS = ["mock"]
# Example when configured:
# PROVIDERS = ["mock", "openai", "anthropic"]

target_request = df_requests.iloc[1].to_dict()

retrieved_docs = retrieve_policies(
    query=" ".join(str(v) for v in target_request.values()),
    policy_docs=policy_docs,
    top_k=2,
)

prompt = build_analysis_prompt(target_request, retrieved_docs)

results = []
for provider in PROVIDERS:
    result = run_llm(
        provider=provider,
        prompt=prompt,
        request_id=target_request["request_id"],
    )
    results.append(result)

for r in results:
    print("=" * 100)
    print(f"Provider: {r.provider} | Model: {r.model} | Latency: {r.latency_seconds}s")
    print("=" * 100)
    print(r.text)
    print()


## 8. Parse and Validate JSON

Structured output is only useful if you validate it.

This section checks:

- Is the output valid JSON?
- Does it include required fields?
- Is the recommendation one of the allowed values?


In [ ]:
REQUIRED_FIELDS = [
    "request_id",
    "one_sentence_summary",
    "decision_recommendation",
    "primary_value_driver",
    "key_risks",
    "missing_information",
    "finance_questions",
    "technical_questions",
    "safety_or_compliance_questions",
    "recommended_next_step",
    "confidence",
]

ALLOWED_RECOMMENDATIONS = {
    "approve",
    "approve_with_conditions",
    "defer",
    "reject",
}

ALLOWED_CONFIDENCE = {
    "low",
    "medium",
    "high",
}


def parse_json_safely(text: str) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    try:
        return json.loads(text), None
    except json.JSONDecodeError as e:
        # Try to recover JSON object from surrounding text.
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0)), None
            except json.JSONDecodeError as e2:
                return None, str(e2)
        return None, str(e)


def validate_output(obj: Dict[str, Any]) -> Dict[str, Any]:
    missing_fields = [field for field in REQUIRED_FIELDS if field not in obj]

    recommendation_valid = obj.get("decision_recommendation") in ALLOWED_RECOMMENDATIONS
    confidence_valid = obj.get("confidence") in ALLOWED_CONFIDENCE

    list_fields = [
        "key_risks",
        "missing_information",
        "finance_questions",
        "technical_questions",
        "safety_or_compliance_questions",
    ]
    list_fields_valid = all(isinstance(obj.get(field), list) for field in list_fields)

    return {
        "missing_fields": missing_fields,
        "recommendation_valid": recommendation_valid,
        "confidence_valid": confidence_valid,
        "list_fields_valid": list_fields_valid,
        "is_valid": (
            len(missing_fields) == 0
            and recommendation_valid
            and confidence_valid
            and list_fields_valid
        ),
    }


validated_rows = []

for r in results:
    parsed, error = parse_json_safely(r.text)
    validation = validate_output(parsed) if parsed else {"is_valid": False, "parse_error": error}

    validated_rows.append(
        {
            "provider": r.provider,
            "model": r.model,
            "request_id": target_request["request_id"],
            "latency_seconds": r.latency_seconds,
            "input_chars": r.input_chars,
            "output_chars": r.output_chars,
            "parse_error": error,
            **validation,
            "parsed_output": parsed,
        }
    )

df_validated = pd.DataFrame(validated_rows)
df_validated[[
    "provider",
    "model",
    "request_id",
    "latency_seconds",
    "is_valid",
    "recommendation_valid",
    "confidence_valid",
    "list_fields_valid",
    "parse_error",
]]


## 9. Simple Evaluation Rubric

This is a lightweight heuristic rubric.  
A real eval system would use:

- Human review
- LLM-as-judge calibrated against humans
- Golden examples
- Regression tests
- Safety and policy-specific checks
- Latency and cost thresholds


In [ ]:
def score_analysis(parsed: Dict[str, Any], request: Dict[str, str]) -> Dict[str, float]:
    if not parsed:
        return {
            "structure_score": 0,
            "risk_score": 0,
            "finance_score": 0,
            "actionability_score": 0,
            "total_score": 0,
        }

    structure_score = 1.0 if all(field in parsed for field in REQUIRED_FIELDS) else 0.0

    risk_score = min(1.0, len(parsed.get("key_risks", [])) / 3)

    finance_terms = {"cost", "budget", "margin", "scenario", "run-rate", "arr", "utilization", "capacity"}
    finance_text = " ".join(parsed.get("finance_questions", [])).lower()
    finance_score = min(1.0, len([t for t in finance_terms if t in finance_text]) / 3)

    next_step = parsed.get("recommended_next_step", "")
    actionability_score = 1.0 if len(next_step.split()) >= 10 else 0.5 if next_step else 0.0

    total_score = round(
        0.30 * structure_score
        + 0.25 * risk_score
        + 0.25 * finance_score
        + 0.20 * actionability_score,
        3,
    )

    return {
        "structure_score": round(structure_score, 3),
        "risk_score": round(risk_score, 3),
        "finance_score": round(finance_score, 3),
        "actionability_score": round(actionability_score, 3),
        "total_score": total_score,
    }


score_rows = []
for row in validated_rows:
    parsed = row["parsed_output"]
    scores = score_analysis(parsed, target_request)
    score_rows.append({
        "provider": row["provider"],
        "model": row["model"],
        "request_id": row["request_id"],
        **scores,
    })

df_scores = pd.DataFrame(score_rows)
df_scores


## 10. Cost Template

Token pricing changes over time and differs by model.  
For a public GitHub notebook, keep pricing user-configurable.

Fill in the rates for your selected models before using this for real analysis.


In [ ]:
# User-configurable pricing placeholders.
# Replace with the actual input/output price per 1M tokens for the model you are testing.
MODEL_PRICING = {
    "mock-analyst-v1": {"input_per_1m_tokens": 0.0, "output_per_1m_tokens": 0.0},
    # "your-openai-model": {"input_per_1m_tokens": 0.00, "output_per_1m_tokens": 0.00},
    # "your-anthropic-model": {"input_per_1m_tokens": 0.00, "output_per_1m_tokens": 0.00},
}

def rough_token_estimate_from_chars(char_count: int) -> int:
    # Very rough English estimate: 1 token ≈ 4 characters.
    return max(1, math.ceil(char_count / 4))


def estimate_cost(model: str, input_chars: int, output_chars: int) -> Dict[str, float]:
    pricing = MODEL_PRICING.get(
        model,
        {"input_per_1m_tokens": 0.0, "output_per_1m_tokens": 0.0},
    )

    input_tokens = rough_token_estimate_from_chars(input_chars)
    output_tokens = rough_token_estimate_from_chars(output_chars)

    input_cost = input_tokens / 1_000_000 * pricing["input_per_1m_tokens"]
    output_cost = output_tokens / 1_000_000 * pricing["output_per_1m_tokens"]

    return {
        "estimated_input_tokens": input_tokens,
        "estimated_output_tokens": output_tokens,
        "estimated_total_cost_usd": round(input_cost + output_cost, 6),
    }


cost_rows = []
for r in results:
    cost_rows.append({
        "provider": r.provider,
        "model": r.model,
        "request_id": target_request["request_id"],
        **estimate_cost(r.model, r.input_chars, r.output_chars),
    })

df_costs = pd.DataFrame(cost_rows)
df_costs


## 11. Run the Full Batch

This section runs all synthetic requests through the selected providers and creates a model-eval table.


In [ ]:
all_eval_rows = []

for _, request_row in df_requests.iterrows():
    req = request_row.to_dict()
    retrieved_docs = retrieve_policies(
        query=" ".join(str(v) for v in req.values()),
        policy_docs=policy_docs,
        top_k=2,
    )
    prompt = build_analysis_prompt(req, retrieved_docs)

    for provider in PROVIDERS:
        result = run_llm(provider, prompt, req["request_id"])
        parsed, error = parse_json_safely(result.text)
        validation = validate_output(parsed) if parsed else {"is_valid": False}
        scores = score_analysis(parsed, req)
        cost = estimate_cost(result.model, result.input_chars, result.output_chars)

        all_eval_rows.append(
            {
                "request_id": req["request_id"],
                "request_type": req["request_type"],
                "provider": result.provider,
                "model": result.model,
                "latency_seconds": result.latency_seconds,
                "is_valid": validation["is_valid"],
                "decision_recommendation": parsed.get("decision_recommendation") if parsed else None,
                "confidence": parsed.get("confidence") if parsed else None,
                **scores,
                **cost,
                "one_sentence_summary": parsed.get("one_sentence_summary") if parsed else None,
                "recommended_next_step": parsed.get("recommended_next_step") if parsed else None,
            }
        )

df_eval = pd.DataFrame(all_eval_rows)
df_eval


## 12. Executive Briefing Generator

This creates a concise leadership-ready summary from the structured results.


In [ ]:
def build_exec_brief(df_eval: pd.DataFrame) -> str:
    lines = []
    lines.append("# Executive Briefing: LLM Investment Request Triage")
    lines.append("")
    lines.append("## Summary")
    lines.append(
        f"- Reviewed {df_eval['request_id'].nunique()} synthetic AI infrastructure / finance requests."
    )
    lines.append(
        f"- Providers tested: {', '.join(sorted(df_eval['provider'].unique()))}."
    )
    lines.append(
        f"- Average eval score: {df_eval['total_score'].mean():.2f}."
    )
    lines.append("")

    lines.append("## Recommendations by Request")
    for _, row in df_eval.iterrows():
        lines.append(
            f"- **{row['request_id']} / {row['request_type']}**: "
            f"{row['decision_recommendation']} — {row['one_sentence_summary']}"
        )

    lines.append("")
    lines.append("## Operating Takeaways")
    lines.append("- Structured JSON output makes model responses easier to validate and compare.")
    lines.append("- Retrieval helps ground the model in policy context instead of relying only on the prompt.")
    lines.append("- Evaluation should combine structure checks, quality scoring, latency, cost, and human review.")
    lines.append("- For production, replace mock mode with controlled OpenAI / Anthropic calls and add regression tests.")

    return "\n".join(lines)


exec_brief = build_exec_brief(df_eval)
print(exec_brief)


## 13. Export Results

The CSV output can be used for a GitHub project artifact, dashboard, or follow-up analysis.


In [ ]:
output_path = "llm_eval_results.csv"
df_eval.to_csv(output_path, index=False)
print(f"Saved: {output_path}")


## 14. How to Turn This Into a Full Portfolio Project

Recommended repo structure:

```text
llm-finance-eval-assistant/
├── README.md
├── notebooks/
│   └── 01_llm_model_eval_finance_ops.ipynb
├── data/
│   └── synthetic_requests.csv
├── src/
│   ├── providers.py
│   ├── prompts.py
│   ├── retrieval.py
│   ├── evaluation.py
│   └── cost.py
├── outputs/
│   └── llm_eval_results.csv
└── requirements.txt
```

Potential extensions:

1. Add real embeddings and vector search
2. Add a Streamlit dashboard
3. Add model-to-model comparison between OpenAI and Anthropic
4. Add human review labels and calibration
5. Add prompt regression tests
6. Add cost and latency charts
7. Add a model-routing recommendation engine
8. Add safety / compliance red-flag detection
9. Add an executive memo generator
10. Add scenario analysis for inference cost reduction

Good project title:

**LLM Finance & Infrastructure Evaluation Assistant**

Good resume bullet:

> Built an end-to-end LLM evaluation workflow for AI infrastructure investment requests, comparing model output quality, structured-output reliability, latency, and cost using synthetic finance and policy data; implemented retrieval-grounded prompting, JSON validation, scoring rubric, and executive briefing generation.


## 15. References

- OpenAI API platform: https://platform.openai.com/
- Anthropic Claude API docs: https://docs.anthropic.com/
- Anthropic Messages API: https://docs.anthropic.com/en/api/messages
- Anthropic client SDKs: https://docs.anthropic.com/en/api/client-sdks
